# NBA 2024–2025 Player Performance Forecasting (Next-Game Regression)

This notebook builds an end-to-end **player next-game performance forecasting** model using game-level player stats.

## What it does
- Loads a game-by-game player dataset (2024–2025 season)
- Creates a **next-game target** (e.g., fantasy points next game)
- Engineers **pre-game features** (rolling form, fatigue, context)
- Trains baseline + ML models with a **time-based split**
- Evaluates using MAE/RMSE and shows feature importance

> **Important (no leakage):** Features for game *t* only use information available **before** game *t* (prior games).

> **Environment:** Run the **Setup (install libraries)** cell below once. It installs a NumPy 2.x stack and upgrades streamlit, scanpy, astropy, and other conflicting packages to compatible versions. Then **Kernel → Restart** and run the notebook from the top.


### Setup (install libraries) — run once, then Kernel → Restart

1. **Notebook stack:** NumPy 2.x, pandas 3.x, scikit-learn, pyarrow, matplotlib, shap.  
2. **Upgrade conflicting packages** to NumPy 2–compatible versions: streamlit, scanpy, astropy, pywavelets, fcsparser, gensim (+ FuzzyTM), pygam.  

After the cell finishes, **restart the kernel** so the new packages are loaded, then run the notebook from the top.

In [1]:
# # 1) Notebook stack (NumPy 2.x)
# %pip install --upgrade "numpy>=2.0" "pandas>=3.0" "scikit-learn>=1.4" "pyarrow>=14" "numexpr>=2.8" "bottleneck>=1.3.7" "matplotlib>=3.8" "shap>=0.44"

# # 2) Upgrade previously conflicting packages to NumPy 2–compatible versions
# %pip install --upgrade "streamlit>=1.31" "scanpy" "astropy>=6.1" "pywavelets>=1.6" "fcsparser" "pygam"
# %pip install --upgrade "FuzzyTM>=0.4.0" "gensim"

# print("\n>>> Restart the kernel now (Kernel → Restart), then run the notebook from the top.")

In [2]:
# --- Configuration ---
from pathlib import Path
# Resolve path: run from project root (NBAIntelligence) or from resources/notebooks
_root = Path("dist/data/database_24_25.csv")
DATA_PATH = str(_root) if _root.exists() else str(Path("../../dist/data/database_24_25.csv").resolve())

# Column names (matching database_24_25.csv: Player, Tm, Opp, Data, PTS, TRB, etc.)
COL_PLAYER_ID = "Player"      # player name as identifier (no ID in CSV)
COL_PLAYER_NAME = "Player"
COL_GAME_ID = None            # not in dataset
COL_DATE = "Data"             # game date YYYY-MM-DD
COL_TEAM = "Tm"               # e.g. "LAL"
COL_OPP = "Opp"               # opponent
COL_HOME_AWAY = None          # not in dataset

# Box score columns (match CSV headers)
COL_PTS = "PTS"
COL_REB = "TRB"
COL_AST = "AST"
COL_STL = "STL"
COL_BLK = "BLK"
COL_TOV = "TOV"

COL_MIN = "MP"                # minutes played


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor


In [4]:
# --- Load data ---
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()


(16512, 25)


,Player,Tm,Opp,Res,MP,FG,FGA,FG%,3P,3PA,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,GmSc,Data
0,Jayson Tatum,BOS,NYK,W,30.30,14,18,0.778,8,11,...,4,4,10,1,1,1,1,37,38.1,2024-10-22
1,Anthony Davis,LAL,MIN,W,37.58,11,23,0.478,1,3,...,13,16,4,1,3,1,1,36,34.0,2024-10-22
2,Derrick White,BOS,NYK,W,26.63,8,13,0.615,6,10,...,3,3,4,1,0,0,1,24,22.4,2024-10-22
3,Jrue Holiday,BOS,NYK,W,30.52,7,9,0.778,4,6,...,2,4,4,1,0,0,2,18,19.5,2024-10-22
4,Miles McBride,NYK,BOS,L,25.85,8,10,0.800,4,5,...,0,0,2,0,0,1,1,22,17.8,2024-10-22


## 1) Basic cleaning + date parsing

Adjust column mappings above if your dataset uses different names.


In [5]:
# Parse date
df[COL_DATE] = pd.to_datetime(df[COL_DATE])

# Coerce box-score columns to numeric (CSV may have mixed types)
for c in [COL_PTS, COL_REB, COL_AST, COL_STL, COL_BLK, COL_TOV, COL_MIN]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Sort for time-aware feature engineering
sort_cols = [COL_PLAYER_ID, COL_DATE]
if COL_GAME_ID and COL_GAME_ID in df.columns:
    sort_cols.append(COL_GAME_ID)
df = df.sort_values(sort_cols).reset_index(drop=True)

# (Optional) standardize home/away if you have it
if COL_HOME_AWAY and COL_HOME_AWAY in df.columns:
    df[COL_HOME_AWAY] = df[COL_HOME_AWAY].astype(str).str.upper().str.strip()

df[[COL_PLAYER_ID, COL_DATE]].head()


,Player,Data
0,A.J. Green,2024-10-23
1,A.J. Green,2024-10-25
2,A.J. Green,2024-10-27
3,A.J. Green,2024-10-28
4,A.J. Green,2024-10-31


## 2) Define the target: next-game fantasy score

Fantasy score formula (DraftKings-style, without double-double bonuses by default):

- PTS: 1.0  
- REB: 1.25  
- AST: 1.5  
- STL: 2.0  
- BLK: 2.0  
- TOV: -0.5  

You can customize weights as needed.


In [6]:
def draftkings_fantasy_score(row) -> float:
    return (
        1.0 * row[COL_PTS]
        + 1.25 * row[COL_REB]
        + 1.5 * row[COL_AST]
        + 2.0 * row[COL_STL]
        + 2.0 * row[COL_BLK]
        - 0.5 * row[COL_TOV]
    )

# Compute current-game fantasy score
df["fantasy_score"] = df.apply(draftkings_fantasy_score, axis=1)

# Target: next-game fantasy score per player
df["y_next_fantasy"] = df.groupby(COL_PLAYER_ID)["fantasy_score"].shift(-1)
# Multi-target for next step (4): next-game PTS, AST, REB (then derive fantasy)
df["y_next_pts"] = df.groupby(COL_PLAYER_ID)[COL_PTS].shift(-1)
df["y_next_ast"] = df.groupby(COL_PLAYER_ID)[COL_AST].shift(-1)
df["y_next_reb"] = df.groupby(COL_PLAYER_ID)[COL_REB].shift(-1)

df[[COL_PLAYER_ID, COL_DATE, "fantasy_score", "y_next_fantasy", "y_next_pts", "y_next_ast", "y_next_reb"]].head(10)


,Player,Data,fantasy_score,y_next_fantasy,y_next_pts,y_next_ast,y_next_reb
0,A.J. Green,2024-10-23,3.50,11.75,9.0,1.0,1.0
1,A.J. Green,2024-10-25,11.75,6.00,5.0,1.0,0.0
2,A.J. Green,2024-10-27,6.00,4.25,3.0,0.0,1.0
3,A.J. Green,2024-10-28,4.25,0.00,0.0,0.0,0.0
4,A.J. Green,2024-10-31,0.00,29.50,21.0,1.0,6.0
5,A.J. Green,2024-11-04,29.50,16.50,12.0,2.0,0.0
6,A.J. Green,2024-11-07,16.50,9.00,9.0,0.0,0.0
7,A.J. Green,2024-11-08,9.00,21.75,12.0,0.0,3.0
8,A.J. Green,2024-11-10,21.75,26.25,12.0,3.0,5.0
9,A.J. Green,2024-11-12,26.25,6.00,3.0,1.0,2.0


## 3) Feature engineering (pre-game features only)

We create rolling features based on *prior* games:
- Rolling means and stds for recent form (3/5/10 games)
- Rolling minutes (if available)
- Fatigue features: days since last game, games in last 7 days
- Simple context: home/away, opponent/team IDs (categorical)

All rolling features use `.shift(1)` to ensure they only include games **before** the current game.


In [7]:
# Helper to create lagged rolling stats (no leakage)
def add_rolling_features(df_in: pd.DataFrame, col: str, windows=(3,5,10)) -> pd.DataFrame:
    g = df_in.groupby(COL_PLAYER_ID)[col]
    for w in windows:
        df_in[f"{col}_roll_mean_{w}"] = g.shift(1).rolling(window=w, min_periods=1).mean()
        df_in[f"{col}_roll_std_{w}"]  = g.shift(1).rolling(window=w, min_periods=1).std()
    # last game value (lag 1)
    df_in[f"{col}_lag_1"] = g.shift(1)
    return df_in

# Rolling performance features
for c in [COL_PTS, COL_REB, COL_AST, COL_STL, COL_BLK, COL_TOV, "fantasy_score"]:
    df = add_rolling_features(df, c)

# Minutes rolling if available
if COL_MIN in df.columns:
    df = add_rolling_features(df, COL_MIN)


In [8]:
# Fatigue features
df["prev_game_date"] = df.groupby(COL_PLAYER_ID)[COL_DATE].shift(1)
df["days_since_last_game"] = (df[COL_DATE] - df["prev_game_date"]).dt.days

# Games in last 7 days (count prior games within 7-day lookback)
# We'll compute using rolling window on dates per player.
def games_in_last_n_days(group: pd.DataFrame, n_days: int = 7) -> pd.Series:
    dates = group[COL_DATE].values.astype("datetime64[ns]")
    out = np.zeros(len(group), dtype=float)
    for i in range(len(group)):
        # lookback up to current date, excluding current game (so use i-1)
        if i == 0:
            out[i] = 0
            continue
        start = dates[i] - np.timedelta64(n_days, "D")
        # count previous games with date >= start
        out[i] = np.sum((dates[:i] >= start))
    return pd.Series(out, index=group.index)

df["games_last_7_days"] = df.groupby(COL_PLAYER_ID, group_keys=False).apply(games_in_last_n_days, n_days=7)

df[["days_since_last_game","games_last_7_days"]].describe()


/var/folders/g3/fb8fscz10w57_t0_p147_7dw0000gn/T/ipykernel_15086/2918628857.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df["games_last_7_days"] = df.groupby(COL_PLAYER_ID, group_keys=False).apply(games_in_last_n_days, n_days=7)


,days_since_last_game,games_last_7_days
count,15950.000000,16512.000000
mean,2.949404,2.487706
std,3.938172,1.144131
min,1.000000,0.000000
25%,2.000000,2.000000
50%,2.000000,3.000000
75%,3.000000,3.000000
max,89.000000,4.000000


### 3b) Opponent/team strength features (Next step 1)

Add rolling **team scoring** and **opponent defensive strength** (avg PTS allowed) using only prior games.

In [9]:
# Game-level: team PTS per game (mean of players) and opponent PTS allowed per game
team_pts_per_game = df.groupby([COL_TEAM, COL_DATE])[COL_PTS].mean().reset_index()
team_pts_per_game.columns = [COL_TEAM, COL_DATE, "team_pts_game"]
opp_pts_allowed = df.groupby([COL_OPP, COL_DATE])[COL_PTS].sum().reset_index()
opp_pts_allowed.columns = [COL_OPP, COL_DATE, "opp_pts_allowed_game"]

# Rolling (lagged) team strength and opponent defensive strength
for g in [COL_TEAM, COL_DATE]:
    if g not in team_pts_per_game.columns:
        raise ValueError(g)
team_pts_per_game = team_pts_per_game.sort_values([COL_TEAM, COL_DATE]).reset_index(drop=True)
opp_pts_allowed = opp_pts_allowed.sort_values([COL_OPP, COL_DATE]).reset_index(drop=True)

team_pts_per_game["team_pts_roll5"] = (
    team_pts_per_game.groupby(COL_TEAM)["team_pts_game"].shift(1).rolling(5, min_periods=1).mean()
)
opp_pts_allowed["opp_def_roll5"] = (
    opp_pts_allowed.groupby(COL_OPP)["opp_pts_allowed_game"].shift(1).rolling(5, min_periods=1).mean()
)

# Merge back to player-level df (one row per player-game)
df = df.merge(
    team_pts_per_game[[COL_TEAM, COL_DATE, "team_pts_roll5"]],
    on=[COL_TEAM, COL_DATE],
    how="left"
)
df = df.merge(
    opp_pts_allowed[[COL_OPP, COL_DATE, "opp_def_roll5"]],
    on=[COL_OPP, COL_DATE],
    how="left"
)
df[["team_pts_roll5", "opp_def_roll5"]].describe()

,team_pts_roll5,opp_def_roll5
count,16503.000000,16502.000000
mean,10.709122,113.249300
std,1.020729,6.701679
min,8.148046,89.400000
25%,9.989697,108.800000
50%,10.630427,113.200000
75%,11.387273,117.600000
max,14.113889,133.800000


## 4) Build the modeling table

We drop rows where the target is missing (last game per player has no next-game).
We also drop rows where key pre-game features are missing (e.g., first ever game has no lag features).


In [10]:
# Keep only rows with target
model_df = df.dropna(subset=["y_next_fantasy"]).copy()

# Basic filtering: remove players with too few games if you want
# model_df = model_df.groupby(COL_PLAYER_ID).filter(lambda x: len(x) >= 10)

# Choose feature columns
numeric_features = [c for c in model_df.columns if c.endswith(("lag_1","_roll_mean_3","_roll_std_3","_roll_mean_5","_roll_std_5","_roll_mean_10","_roll_std_10"))]
numeric_features += ["days_since_last_game", "games_last_7_days", "team_pts_roll5", "opp_def_roll5"]

categorical_features = []
for c in [COL_TEAM, COL_OPP, COL_HOME_AWAY]:
    if c is not None and c in model_df.columns:
        categorical_features.append(c)

# Optional: add player_id as a categorical to let the model learn player baselines
categorical_features += [COL_PLAYER_ID]

target_col = "y_next_fantasy"

X = model_df[numeric_features + categorical_features].copy()
y = model_df[target_col].copy()

X.shape, y.shape, len(numeric_features), len(categorical_features)


((15950, 63), (15950,), 60, 3)

## 5) Time-based split (recommended)

We split by date to simulate real forecasting:
- Train on earlier games
- Test on later games

Adjust the cutoff date depending on how your season is represented.


In [38]:
# Split: (1) Last game of each team = test (inference); (2) Rest = 80% train / 20% validation by time
last_date_per_team = model_df.groupby(COL_TEAM)[COL_DATE].transform("max")
test_mask = model_df[COL_DATE] == last_date_per_team

train_val_df = model_df[~test_mask].copy()
test_df = model_df[test_mask].copy()

# 80:20 time-based split for train vs validation (on non-test data)
cutoff = train_val_df[COL_DATE].quantile(0.8)
train_mask = train_val_df[COL_DATE] <= cutoff
val_mask = train_val_df[COL_DATE] > cutoff
val_df = train_val_df[val_mask]

X_train = X.loc[train_val_df.index[train_mask]]
y_train = y.loc[train_val_df.index[train_mask]]
X_val   = X.loc[train_val_df.index[val_mask]]
y_val   = y.loc[train_val_df.index[val_mask]]
X_test  = X.loc[test_df.index]
y_test  = y.loc[test_df.index]

print("Train:", X_train.shape, "Val:", X_val.shape, "Test (last game per team):", X_test.shape)
print("Cutoff (train/val):", cutoff)


Train: (12556, 63) Val: (3123, 63) Test (last game per team): (271, 63)
Cutoff (train/val): 2025-01-14 00:00:00


## 6) Baseline model (rolling mean)

A strong baseline is the player's rolling mean fantasy score over the last 5 games.


In [13]:
# Baseline prediction: fantasy_score_roll_mean_5 (prior games)
baseline_col = "fantasy_score_roll_mean_5"
if baseline_col not in X.columns:
    raise ValueError(f"Baseline column {baseline_col} not found. Check feature engineering.")

y_val_base = X_val[baseline_col].values
y_pred_base = X_test[baseline_col].values
mae_base = mean_absolute_error(y_val, y_val_base)
rmse_base = mean_squared_error(y_val, y_val_base)

mae_base, rmse_base


(7.733258618849397, 99.21937262069947)

## 7) ML model: Gradient Boosting (tabular-friendly)

We use `HistGradientBoostingRegressor` (fast, strong baseline in sklearn).
Categoricals are one-hot encoded.


In [15]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

gbr = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=400,
    random_state=42
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", gbr)
])

model.fit(X_train, y_train)

y_val_hgb = model.predict(X_val)
y_pred_hgb = model.predict(X_test)
mae_hgb = mean_absolute_error(y_val, y_val_hgb)
rmse_hgb = mean_squared_error(y_val, y_val_hgb)
r2_hgb = r2_score(y_val, y_val_hgb)

{"baseline_mae": mae_base, "mae": mae_hgb, "baseline_rmse": rmse_base, "rmse": rmse_hgb, "r2": r2_hgb}


{'baseline_mae': 7.733258618849397,
 'mae': 7.613637177447393,
 'baseline_rmse': 99.21937262069947,
 'rmse': 93.70321318920888,
 'r2': 0.5702297232831869}

## 8) Optional: Random Forest (compare)

Sometimes useful as another baseline. Usually boosting wins for this kind of problem.


## 7b) Multi-target (PTS, AST, REB) then derive fantasy (Next step 4)

Train separate models for next-game PTS, AST, and REB; then combine predictions with rolling proxies for STL/BLK/TOV to get fantasy score.

In [17]:
# # Targets for multi-output (same rows as model_df)
# y_pts = model_df["y_next_pts"]
# y_ast = model_df["y_next_ast"]
# y_reb = model_df["y_next_reb"]

# # Train one model per target (same features, same preprocess)
# model_pts = Pipeline(steps=[("preprocess", preprocess), ("model", HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=400, random_state=42))])
# model_ast = Pipeline(steps=[("preprocess", preprocess), ("model", HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=400, random_state=42))])
# model_reb = Pipeline(steps=[("preprocess", preprocess), ("model", HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=400, random_state=42))])

# model_pts.fit(X_train, y_train.loc[X_train.index])
# model_ast.fit(X_train, y_ast.loc[X_train.index])
# model_reb.fit(X_train, y_reb.loc[X_train.index])

# pred_pts = model_pts.predict(X_test)
# pred_ast = model_ast.predict(X_test)
# pred_reb = model_reb.predict(X_test)

# # Derive fantasy from predicted PTS/AST/REB + rolling proxies for STL, BLK, TOV (DraftKings formula)
# stl_proxy = X_test[f"{COL_STL}_roll_mean_5"].values if f"{COL_STL}_roll_mean_5" in X_test.columns else np.zeros(len(X_test))
# blk_proxy = X_test[f"{COL_BLK}_roll_mean_5"].values if f"{COL_BLK}_roll_mean_5" in X_test.columns else np.zeros(len(X_test))
# tov_proxy = X_test[f"{COL_TOV}_roll_mean_5"].values if f"{COL_TOV}_roll_mean_5" in X_test.columns else np.zeros(len(X_test))

# y_pred_fantasy_multi = 1.0 * pred_pts + 1.25 * pred_reb + 1.5 * pred_ast + 2.0 * stl_proxy + 2.0 * blk_proxy - 0.5 * tov_proxy

# mae_multi = mean_absolute_error(y_test, y_pred_fantasy_multi)
# rmse_multi = mean_squared_error(y_test, y_pred_fantasy_multi)
# print("Multi-target (PTS+AST+REB → fantasy): MAE =", round(mae_multi, 4), "RMSE =", round(rmse_multi, 4))
# print("Single-target fantasy: MAE =", round(mae, 4), "RMSE =", round(rmse, 4))

# res["Multi-target-HGB"] = y_pred_fantasy_multi 

## 7c) Explainability with SHAP (Next step 5)

Use SHAP to interpret the single-target fantasy model. Install with: `pip install shap`

In [18]:
# try:
#     import shap

#     # Transform features (sample for speed)
#     n_sample = min(500, len(X_train))
#     X_sample = X_train.iloc[:n_sample]
#     X_transformed = model.named_steps["preprocess"].transform(X_sample)
#     if hasattr(X_transformed, "toarray"):
#         X_transformed = X_transformed.toarray()

#     # TreeExplainer for the inner GBR
#     inner = model.named_steps["model"]
#     explainer = shap.TreeExplainer(inner, X_transformed)
#     shap_vals = explainer.shap_values(X_transformed)

#     # Feature names after preprocessing (numeric + one-hot cat)
#     num_names = numeric_features
#     cat_ohe = model.named_steps["preprocess"].named_transformers_["cat"].named_steps["onehot"]
#     cat_names = cat_ohe.get_feature_names_out(categorical_features).tolist()
#     all_names = num_names + cat_names

#     shap.summary_plot(shap_vals, X_transformed, feature_names=all_names, max_display=15, show=False)
#     plt.title("SHAP summary (single-target fantasy model)")
#     plt.tight_layout()
#     plt.show()
# except ImportError:
#     print("Install shap: pip install shap")

## 8) Other models

In [16]:
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42
)

rf_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", rf),
])

rf_model.fit(X_train, y_train)

y_val_rf = rf_model.predict(X_val)  
y_pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_val, y_val_rf)
rmse_rf = mean_squared_error(y_val, y_val_rf)

{"baseline_mae": mae_base, "rf_mae": mae_rf, "baseline_rmse": rmse_base, "rf_rmse": rmse_rf}


{'baseline_mae': 7.733258618849397,
 'rf_mae': 7.603647094444621,
 'baseline_rmse': 99.21937262069947,
 'rf_rmse': 93.34386705099693}

### 8b) Other models: XGBoost, LightGBM, CatBoost, TabNet

Train and compare XGBoost, LightGBM, CatBoost, and PyTorch TabNet using the same preprocessed features. Install with: `pip install xgboost lightgbm catboost pytorch-tabnet`

In [19]:
# Uncomment to install: %pip install xgboost lightgbm catboost pytorch-tabnet
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def mape(y_true, y_pred, eps=1e-8):
    """Mean Absolute Percentage Error (%)."""
    return 100 * np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + eps)))

# Preprocess once to dense arrays (same as in pipeline)
X_train_t = model.named_steps["preprocess"].transform(X_train)
X_val_t = model.named_steps["preprocess"].transform(X_val)
X_test_t = model.named_steps["preprocess"].transform(X_test)
if hasattr(X_train_t, "toarray"):
    X_train_t = X_train_t.toarray()
    X_test_t = X_test_t.toarray()
y_train_vals = np.asarray(y_train)
y_val_vals = np.asarray(y_val)
y_test_vals = np.asarray(y_test)

# Results from previous sections: Rolling5 (baseline), HistGradientBoosting, RandomForest — with MAPE
y_pred_roll5 = X_test["fantasy_score_roll_mean_5"].values
y_pred_gbr = model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

results_8b = {}
results_8b["Rolling5 (baseline)"] = (mae_base, rmse_base, mape(y_test_vals, y_pred_roll5))
results_8b["HistGradientBoosting"] = (mae_hgb, rmse_hgb, mape(y_val_vals, y_val_hgb))
results_8b["RandomForest"] = (mae_rf, rmse_rf, mape(y_test_vals, y_pred_rf))

# XGBoost
import xgboost as xgb
xgb_model = xgb.XGBRegressor(max_depth=6, learning_rate=0.05, n_estimators=400, random_state=42, n_jobs=-1)
xgb_model.fit(X_train_t, y_train_vals)

y_val_xgb = xgb_model.predict(X_val_t)
y_pred_xgb = xgb_model.predict(X_test_t)
results_8b["XGBoost"] = (
    mean_absolute_error(y_val_vals, y_val_xgb),
    mean_squared_error(y_val_vals, y_val_xgb),
    mape(y_val_vals, y_val_xgb),
)

In [20]:
# LightGBM

import lightgbm as lgb
lgb_model = lgb.LGBMRegressor(max_depth=6, learning_rate=0.05, n_estimators=400, random_state=42, n_jobs=-1, verbose=-1)
lgb_model.fit(X_train_t, y_train_vals)

y_val_lgb = lgb_model.predict(X_val_t)  
y_pred_lgb = lgb_model.predict(X_test_t)
results_8b["LightGBM"] = (
    mean_absolute_error(y_val_vals, y_val_lgb),
    mean_squared_error(y_val_vals, y_val_lgb),
    mape(y_val_vals, y_val_lgb),
)

In [21]:
# CatBoost
import catboost as cb
cb_model = cb.CatBoostRegressor(depth=6, learning_rate=0.05, iterations=400, random_state=42, verbose=0)
cb_model.fit(X_train_t, y_train_vals)

y_val_cb = cb_model.predict(X_val_t)
y_pred_cb = cb_model.predict(X_test_t)
results_8b["CatBoost"] = (
    mean_absolute_error(y_val_vals, y_val_cb),
    mean_squared_error(y_val_vals, y_val_cb),
    mape(y_val_vals, y_val_cb),
)

In [22]:
# PyTorch TabNet (slower; smaller epochs for speed)
from pytorch_tabnet.tab_model import TabNetRegressor
tabnet = TabNetRegressor(optimizer_params=dict(lr=0.01), seed=42, verbose=0)
tabnet.fit(X_train_t, y_train_vals.reshape(-1, 1), eval_set=[(X_test_t, y_test_vals.reshape(-1, 1))], max_epochs=100, batch_size=1024)

y_val_tabnet = tabnet.predict(X_val_t).flatten()
y_pred_tabnet = tabnet.predict(X_test_t).flatten()
results_8b["TabNet"] = (
    mean_absolute_error(y_val_vals, y_val_tabnet),
    mean_squared_error(y_val_vals, y_val_tabnet),
    mape(y_val_vals, y_val_tabnet),
)


Early stopping occurred at epoch 56 with best_epoch = 46 and best_val_0_mse = 90.56895


In [23]:
# Comparison table (MAE, RMSE, MAPE) — includes Rolling5, HistGradientBoosting, RandomForest from previous sections
comparison = pd.DataFrame([
    {"Model": k, "MAE": v[0], "RMSE": v[1], "MAPE (%)": v[2]} for k, v in results_8b.items()
])
print(comparison.to_string(index=False))

               Model      MAE      RMSE     MAPE (%)
 Rolling5 (baseline) 7.733259 99.219373 3.007381e+08
HistGradientBoosting 7.613637 93.703213 2.584341e+09
        RandomForest 7.603647 93.343867 6.162230e+08
             XGBoost 7.566847 93.186354 2.465129e+09
            LightGBM 7.567924 93.297509 2.512175e+09
            CatBoost 7.542462 92.030991 2.498113e+09
              TabNet 7.752205 97.137216 2.757851e+09


## Export to `public/data/`

This cell writes the model outputs to JSON files for the ML Game Predictor UI:
- `ml_fantasy_export.json` — models, champion, predictions
- `ml_teams_export.json` — team export
- `ml_player_series_export.json` — player series export

**Run this cell** after the comparison table to generate the export files.

In [47]:
# --- Export data for ML Game Predictor UI ---
import json
from pathlib import Path

# Resolve path: run from project root or from resources/notebooks
_root = Path("public/data")
OUT_DIR = _root if _root.exists() else (Path("..") / ".." / "public" / "data").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Champion = model with best MAE
champion_name = min(results_8b, key=lambda k: results_8b[k][0])

# Build validation predictions for each model (y_pred, y_true per row)
y_val_vals = np.asarray(y_val)
val_pred_cols = {
    "Rolling5 (baseline)": X_val["fantasy_score_roll_mean_5"].values,
    "HistGradientBoosting": model.predict(X_val),
    "RandomForest": rf_model.predict(X_val),
    "XGBoost": y_val_xgb,
    "LightGBM": y_val_lgb,
    "CatBoost": y_val_cb,
    "TabNet": y_val_tabnet,
}

val_res = {
    "Rolling5 (baseline)": y_val_base,
    "HistGradientBoosting": y_val_hgb,
    "RandomForest": y_val_rf,
    "XGBoost": y_val_xgb,
    "LightGBM": y_val_lgb,
    "CatBoost": y_val_cb,
    "TabNet": y_val_tabnet,
}

# Build predictions for each model (y_pred, y_true per row)
y_test_vals = np.asarray(y_test)
test_pred_cols = {
    "Rolling5 (baseline)": X_test["fantasy_score_roll_mean_5"].values,
    "HistGradientBoosting": model.predict(X_test),
    "RandomForest": rf_model.predict(X_test),
    "XGBoost": y_pred_xgb,
    "LightGBM": y_pred_lgb,
    "CatBoost": y_pred_cb,
    "TabNet": y_pred_tabnet,
}

model_descriptions = {
    "Rolling5 (baseline)": "Average fantasy score over the last 5 games",
    "HistGradientBoosting": "A tree-based model that builds an ensemble using gradient boosting on histograms",
    "RandomForest": "Ensemble of decision trees using random sampling of features and data",
    "XGBoost": "Extreme Gradient Boosted trees for regression tasks",
    "LightGBM": "Gradient boosting framework that uses tree-based learning algorithms, optimized for speed",
    "CatBoost": "Gradient boosting on decision trees with good support for categorical variables",
    "TabNet": "Deep learning model designed specifically for tabular data",
}

models_export = []
for name, y_pred in test_pred_cols.items():
    mae, rmse, _ = results_8b.get(name, (None, None, None))
    models_export.append({
        "name": name,
        "description": model_descriptions.get(name, ""),
        "mae": float(mae) if mae is not None else None,
        "rmse": float(np.sqrt(rmse)) if rmse is not None else None,
        "predictions": [{"y_pred": float(p), "y_true": float(t)} for p, t in zip(y_pred, y_test_vals)],
    })

# Teams: team -> list of player names (from test set)
teams_export = {}
for _, row in test_df.iterrows():
    team = row[COL_TEAM]
    player = row[COL_PLAYER_ID]
    if team not in teams_export:
        teams_export[team] = []
    if player not in teams_export[team]:
        teams_export[team].append(player)

# Player series: key -> {player, team, games: [{y_pred, y_true, ...}, ...]}
for model_name, champion_val_pred in val_pred_cols.items():
    pred_by_idx = dict(zip(val_df.index, val_res[model_name]))
    # Group by player and team, keeping track of date and index
    grouped = {}
    for idx, row in val_df.iterrows():
        player = row[COL_PLAYER_ID]
        team = row[COL_TEAM]
        date = row[COL_DATE]
        key = f"{player}|{team}"
        if key not in grouped:
            grouped[key] = []
        grouped[key].append((date, idx, player, team))
    player_series_val_export = {}
    for key, games in grouped.items():
        # Sort by date descending and take top 10 games
        top_games = sorted(games, key=lambda x: x[0], reverse=True)[:10]
        player, team = key.split("|", 1)
        player_key_games = []
        for date, idx, _, _ in sorted(top_games, key=lambda x: x[0], reverse=True):
            y_true = float(y_val.loc[idx])
            y_pred = float(pred_by_idx.get(idx, 0))
            player_key_games.append({"y_pred": y_pred, "y_true": y_true, "date": str(date)})
        player_series_val_export[key] = {
            "player": player,
            "team": team,
            "games": player_key_games,
        }
    with open(OUT_DIR / f"ml_player_series_val_export_{model_name}.json", "w") as f:
        json.dump(player_series_val_export, f, indent=2)

# Player series: key -> {player, team, games: [{y_pred, y_true, ...}, ...]}
champion_pred = test_pred_cols[champion_name]
pred_by_idx = dict(zip(test_df.index, champion_pred))
player_series_pred_export = {}
for _, row in test_df.iterrows():
    idx = row.name
    player = row[COL_PLAYER_ID]
    team = row[COL_TEAM]
    key = f"{player}|{team}"
    # y_true = float(y_test.loc[idx])
    y_pred = float(pred_by_idx.get(idx, 0))
    if key not in player_series_pred_export:
        player_series_pred_export[key] = {"player": player, "team": team, "games": []}
    player_series_pred_export[key]["games"].append({"y_pred": y_pred})

with open(OUT_DIR / "ml_fantasy_export.json", "w") as f:
    json.dump({"champion": champion_name, "models": models_export}, f, indent=2)

with open(OUT_DIR / "ml_teams_export.json", "w") as f:
    json.dump(teams_export, f, indent=2)

with open(OUT_DIR / "ml_player_series_pred_export.json", "w") as f:
    json.dump(player_series_pred_export, f, indent=2)



print("Exported to", OUT_DIR)
print("Champion model:", champion_name)

Exported to /Users/jianchenyang/Documents/NBAIntelligence/public/data
Champion model: CatBoost


In [44]:
len(player_series_val_export)

3123

In [ ]:
val_pred_cols

dict_keys(['Rolling5 (baseline)', 'HistGradientBoosting', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost', 'TabNet'])

## 9) Feature importance (approximate)

For tree models inside a pipeline with one-hot encoding, feature names expand.
Below is a helper to extract transformed feature names and plot top importances.

If your model doesn't support `feature_importances_`, you can skip this section.


In [1]:
# import matplotlib.pyplot as plt

# def get_feature_names(preprocessor: ColumnTransformer):
#     feature_names = []
#     # numeric
#     feature_names.extend(numeric_features)
#     # categorical (one-hot)
#     cat_ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
#     cat_names = cat_ohe.get_feature_names_out(categorical_features).tolist()
#     feature_names.extend(cat_names)
#     return feature_names

# # For HistGradientBoostingRegressor, there is no built-in feature_importances_.
# # But for RandomForestRegressor we can show importances:
# rf_est = rf_model.named_steps["model"]
# prep = rf_model.named_steps["preprocess"]

# feat_names = get_feature_names(prep)
# importances = rf_est.feature_importances_

# topk = 25
# idx = np.argsort(importances)[-topk:]
# top_features = [feat_names[i] for i in idx]
# top_importances = importances[idx]

# plt.figure(figsize=(8, 8))
# plt.barh(top_features, top_importances)
# plt.title("Top Feature Importances (Random Forest)")
# plt.xlabel("Importance")
# plt.tight_layout()
# plt.show()


## 10) Next steps (recommended improvements)

- **(1) Done:** Opponent/team strength features (`team_pts_roll5`, `opp_def_roll5`) added in 3b.
- **(4) Done:** Multi-target (PTS, AST, REB) then derive fantasy in section 7b.
- **(5) Done:** SHAP explainability in section 7c (`pip install shap`).

Further ideas:
2. Use **walk-forward validation** by month instead of a single cutoff.
3. Add **player role** features: starter flag, usage rate, season average minutes.
